# 07 · Control Experiment (DFUC Infection)

**This is the notebook that turns a failed attempt into a result.**

Notebooks 02–06 show the derived colour labels do not recover severity. They do
not show the labels are the *only* problem: the architectures, preprocessing or
training configuration could equally be at fault. This notebook settles it by
applying the identical pipeline to a wound classification task that carries
**expert annotation**.

Same architectures. Same preprocessing. Same grouped protocol. Same code. Only
the labels and the output dimension differ. If this works and severity does not,
the limitation is label provenance.

## Scored per photograph, not per image
Predictions are made per image, averaged to one score per `photo_unit`, and only
then turned into metrics. Without that step a photograph with eight augmented
copies counts eight times and one with a single copy counts once, weighting the
test set toward heavily augmented cases. That is the split-leakage error moved to
the scoring stage. The notebook asserts each photograph is scored exactly once.

## Imbalance
1.46:1 is mild, so `pos_weight` in the loss rather than resampling. No SMOTE
here; it belongs in notebook 06's ablation where the imbalance is severe.

## The tissue ablation
Needs `dfuc_tissue.csv` (k-means proportions for DFUC images, same procedure as
notebook 02). If absent, the tissue rows are **flagged and skipped** rather than
silently substituted, and the notebook reports CNN-only.

## Outputs
`results_infection.json`, `ablation_infection.csv`


In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
import torch
torch.set_num_threads(2)

In [2]:
# Cell 1 · config and inputs
from pathlib import Path
import pandas as pd, numpy as np, json, warnings
warnings.filterwarnings('ignore')


INTERIM  = Path('/Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/interim')
FEATURES = Path('/Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/features')
OUT      = Path('/Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs'); OUT.mkdir(exist_ok=True)

FEAT = FEATURES / 'feat_efficientnet_b0_infection.npy'
IDX  = FEATURES / 'index_infection.csv'

SEED, N_FOLDS = 42, 5

missing = [str(p) for p in [FEAT, IDX] if not p.exists()]
if missing:
    print('STOPPING. Missing inputs:'); [print('  -', m) for m in missing]
    print('Run notebook 05 first.')
    raise SystemExit(1)

X   = np.load(FEAT)
idx = pd.read_csv(IDX)
assert len(X) == len(idx), f'feature rows {len(X)} != index rows {len(idx)}'

y    = idx.label.values.astype(int)
fold = idx.fold.values
unit = idx.photo_unit.values

n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())
print(f'images {len(X):,}   units {len(np.unique(unit)):,}   '
      f'patients {idx.patient_id.nunique():,}')
print(f'infected {n_pos:,} / not infected {n_neg:,}  '
      f'= {n_pos/max(n_neg,1):.2f}:1')

images 5,372   units 4,542   patients 1,417
infected 2,767 / not infected 2,605  = 1.06:1


In [3]:
# Cell 2 · tissue proportions for the DFUC images
# TGO-Net needs the same three-channel input here as in notebook 06, so the
# ablation is a like-for-like comparison. If dfuc_tissue.csv is not cached
# yet, it is computed here using the IDENTICAL derivation as notebook 02:
# resize 128, RGB->LAB, k-means k=3 with n_init=4, clusters ordered by
# luminance. Same function, same settings, applied to a different corpus.
#
# One representative image per photo_unit is used (not every copy), to
# keep this affordable inside a control-experiment notebook. This mirrors
# how notebook 06 uses one representative per severity photograph.
TISSUE_CSV = INTERIM / 'dfuc_tissue.csv'
RESIZE, N_INIT_TISSUE = 128, 4

def compute_dfuc_tissue():
    from PIL import Image
    from skimage import color
    from sklearn.cluster import KMeans
    from joblib import Parallel, delayed
    import os, time

    fi = INTERIM / 'folds_infection.csv'
    if not fi.exists():
        print(f'  cannot compute: {fi} not found (need image paths)')
        return None

    df = pd.read_csv(fi)
    rep = df.groupby('photo_unit')['path'].first().reset_index()
    print(f'  computing tissue proportions for {len(rep):,} representative '
          f'DFUC images (one per photo_unit)...')

    def tissue_proportions(path):
        try:
            with Image.open(path) as im:
                a = np.asarray(im.convert('RGB').resize((RESIZE, RESIZE)),
                               dtype=np.float32) / 255.0
        except Exception:
            return np.array([np.nan, np.nan, np.nan])
        lab = color.rgb2lab(a).reshape(-1, 3)
        km = KMeans(3, n_init=N_INIT_TISSUE, random_state=SEED).fit(lab)
        order = np.argsort(km.cluster_centers_[:, 0])
        c = np.bincount(km.labels_, minlength=3)[order].astype(np.float64)
        return c / c.sum()

    n_jobs = max(1, (os.cpu_count() or 2) - 1)
    t0 = time.time()
    rows = Parallel(n_jobs=n_jobs, batch_size=32, verbose=5)(
        delayed(tissue_proportions)(p) for p in rep.path)
    props = np.vstack(rows)
    print(f'  done in {(time.time()-t0)/60:.1f} min')

    out = pd.DataFrame({
        'photo_unit': rep.photo_unit,
        'necrosis_prop': props[:, 0],
        'slough_prop': props[:, 1],
        'granulation_prop': props[:, 2]})
    out = out.dropna()
    out.to_csv(TISSUE_CSV, index=False)
    print(f'  wrote {TISSUE_CSV.resolve()}  ({len(out):,} units)')
    return out

if TISSUE_CSV.exists():
    tis = pd.read_csv(TISSUE_CSV)
    print(f'dfuc_tissue.csv found, cached ({len(tis):,} units)')
else:
    print('dfuc_tissue.csv not found. Computing now (heavy step, one-time).')
    tis = compute_dfuc_tissue()

if tis is not None:
    tis = tis.set_index('photo_unit')
    T = tis.reindex(unit)[['necrosis_prop','slough_prop','granulation_prop']].values
    T = T.astype(np.float32)
    HAVE_TISSUE = not np.isnan(T).any()
    if HAVE_TISSUE:
        assert np.allclose(T.sum(1), 1.0, atol=1e-3), 'proportions do not sum to 1'
        print(f'tissue matrix {T.shape}, aligned and verified')
    else:
        n_missing = int(np.isnan(T).any(axis=1).sum())
        print(f'FLAGGED: {n_missing} images have no tissue proportions '
              f'(their photo_unit was not in the representative set)')
        HAVE_TISSUE = False
        T = np.nan_to_num(T)
else:
    HAVE_TISSUE = False
    T = np.zeros((len(X), 3), dtype=np.float32)
    print('FLAGGED: could not compute tissue proportions.')
    print('  Reporting CNN-only results; tissue rows will be marked skipped.')

dfuc_tissue.csv not found. Computing now (heavy step, one-time).
  computing tissue proportions for 4,542 representative DFUC images (one per photo_unit)...


[Parallel(n_jobs=7)]: Using backend LokyBackend with 7 concurrent workers.
[Parallel(n_jobs=7)]: Done 142 tasks      | elapsed:    3.1s
[Parallel(n_jobs=7)]: Done 1870 tasks      | elapsed:    8.4s
[Parallel(n_jobs=7)]: Done 4378 tasks      | elapsed:   16.9s


  done in 0.3 min
  wrote /Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/interim/dfuc_tissue.csv  (4,542 units)
tissue matrix (5372, 3), aligned and verified


[Parallel(n_jobs=7)]: Done 4542 out of 4542 | elapsed:   17.2s finished


In [4]:
# Cell 3 · TGO-Net, binary head
# Identical to notebook 06 except the output is a single logit instead of
# a CORAL ordinal head. Nothing else changes: same encoder, same gate,
# same initialisation. That identity is what makes this a control.
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(SEED); np.random.seed(SEED)

class TGONetBinary(nn.Module):
    def __init__(s, fdim, use_tissue=True, use_gate=True, p=0.3):
        super().__init__()
        s.ut, s.ug = use_tissue, use_gate
        if use_tissue:
            s.enc = nn.Sequential(nn.Linear(3, 32), nn.LayerNorm(32), nn.GELU(),
                                  nn.Linear(32, 64), nn.LayerNorm(64), nn.GELU())
            if use_gate:
                s.proj  = nn.Linear(64, fdim)
                s.gamma = nn.Parameter(torch.zeros(1))   # starts as pure CNN
                hin = fdim
            else:
                hin = fdim + 64
        else:
            hin = fdim
        s.drop = nn.Dropout(p)
        s.fc   = nn.Linear(hin, 1)

    def forward(s, f, t):
        if s.ut:
            e = s.enc(t)
            if s.ug:
                f = f * (1 + s.gamma * torch.tanh(s.proj(e)))
            else:
                f = torch.cat([f, e], 1)
        return s.fc(s.drop(f)).squeeze(1)

print('TGO-Net (binary) defined')

TGO-Net (binary) defined


In [5]:
# Cell 4 · train one fold, predict per image
from sklearn.metrics import roc_auc_score

def run_fold(k, use_tissue, use_gate, epochs=60, lr=1e-3, bs=256, patience=12):
    te = fold == k
    ids = np.where(~te)[0]
    rng = np.random.default_rng(SEED + k); rng.shuffle(ids)
    n_val = max(1, len(ids) // 6)
    va_i, tr_i = ids[:n_val], ids[n_val:]

    ft = torch.from_numpy(X[tr_i]).float(); tt = torch.from_numpy(T[tr_i]).float()
    yt = torch.from_numpy(y[tr_i]).float()
    fv = torch.from_numpy(X[va_i]).float(); tv = torch.from_numpy(T[va_i]).float()
    fe = torch.from_numpy(X[te]).float();   tev = torch.from_numpy(T[te]).float()

    # class imbalance is mild (1.46:1), so pos_weight in the loss rather
    # than resampling. dtype is explicit: numpy integer division yields
    # float64 and some backends reject it.
    pos_w = torch.tensor([(y[tr_i] == 0).sum() / max((y[tr_i] == 1).sum(), 1)],
                         dtype=torch.float32)

    m = TGONetBinary(X.shape[1], use_tissue, use_gate)
    opt = torch.optim.Adam(m.parameters(), lr=lr, weight_decay=1e-4)

    best, state, wait = -1.0, None, 0
    for _ in range(epochs):
        m.train(); perm = torch.randperm(len(yt))
        for i in range(0, len(yt), bs):
            b = perm[i:i+bs]; opt.zero_grad()
            F.binary_cross_entropy_with_logits(
                m(ft[b], tt[b]), yt[b], pos_weight=pos_w).backward()
            opt.step()
        m.eval()
        with torch.no_grad():
            pv = torch.sigmoid(m(fv, tv)).numpy()
        a = roc_auc_score(y[va_i], pv) if len(np.unique(y[va_i])) > 1 else 0.5
        if a > best:
            best, wait = a, 0
            state = {kk: v.clone() for kk, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break

    m.load_state_dict(state); m.eval()
    with torch.no_grad():
        pe = torch.sigmoid(m(fe, tev)).numpy()
    gam = float(m.gamma.item()) if (use_tissue and use_gate) else None
    return np.where(te)[0], pe, gam

print('fold runner ready')

fold runner ready


In [6]:
# Cell 5 · aggregate per photograph, then score
# Predict per image, average to one score per photo_unit, THEN compute
# metrics. Without this a photograph with eight augmented copies counts
# eight times and one with a single copy counts once, weighting the test
# set toward heavily augmented cases. That is the same class of error as
# the split leakage, moved to the scoring stage.
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             matthews_corrcoef, confusion_matrix)

def aggregate(rows, probs):
    df = pd.DataFrame({'unit': unit[rows], 'p': probs, 'y': y[rows]})
    g = df.groupby('unit').agg(p=('p', 'mean'), y=('y', 'first'))
    return g

def score(name, per_unit, gammas=None, verbose=True):
    yt, pp = per_unit.y.values, per_unit.p.values
    pred = (pp >= 0.5).astype(int)
    tn, fp, fn, tp = confusion_matrix(yt, pred, labels=[0, 1]).ravel()
    res = dict(name=name,
               auroc=float(roc_auc_score(yt, pp)),
               auprc=float(average_precision_score(yt, pp)),
               mcc=float(matthews_corrcoef(yt, pred)),
               sensitivity=float(tp / max(tp + fn, 1)),
               specificity=float(tn / max(tn + fp, 1)),
               n_units=int(len(yt)))
    res['balanced_acc'] = (res['sensitivity'] + res['specificity']) / 2
    if gammas and any(g is not None for g in gammas):
        gs = [g for g in gammas if g is not None]
        res['gamma_per_fold'] = [round(g, 4) for g in gs]
        res['gamma_mean'] = float(np.mean(gs))
        res['gamma_sd']   = float(np.std(gs))
    if verbose:
        print(f'{name}')
        print(f'  AUROC {res["auroc"]:.4f}   AUPRC {res["auprc"]:.4f}   '
              f'MCC {res["mcc"]:.4f}')
        print(f'  sens {res["sensitivity"]:.4f}   spec {res["specificity"]:.4f}'
              f'   balanced acc {res["balanced_acc"]:.4f}   '
              f'units {res["n_units"]:,}')
        if 'gamma_mean' in res:
            print(f'  gamma per fold {res["gamma_per_fold"]}  '
                  f'mean {res["gamma_mean"]:+.4f}  sd {res["gamma_sd"]:.4f}')
    return res

print('scoring ready (per photograph, not per image)')

scoring ready (per photograph, not per image)


In [7]:
# Cell 6 · run the configurations
CONFIGS = [('CNN only', False, False)]
if HAVE_TISSUE:
    CONFIGS += [('+ tissue, concat', True, False),
                ('+ tissue, gated (TGO-Net)', True, True)]
else:
    print('tissue configurations skipped (no dfuc_tissue.csv)\n')

results, per_fold_auroc, unit_preds = [], {}, {}
for name, ut, ug in CONFIGS:
    rows_all, probs_all, gammas, fold_auc = [], [], [], []
    for k in range(N_FOLDS):
        rows, pe, gam = run_fold(k, ut, ug)
        rows_all.append(rows); probs_all.append(pe); gammas.append(gam)
        pu = aggregate(rows, pe)
        fold_auc.append(float(roc_auc_score(pu.y, pu.p))
                        if pu.y.nunique() > 1 else float('nan'))
    rows_all = np.concatenate(rows_all); probs_all = np.concatenate(probs_all)
    pu = aggregate(rows_all, probs_all)
    # every photograph scored exactly once
    assert len(pu) == len(np.unique(unit)), 'unit count mismatch after aggregation'
    unit_preds[name] = pu
    r = score(name, pu, gammas)
    r['per_fold_auroc'] = [round(a, 4) for a in fold_auc]
    print(f'  per fold {r["per_fold_auroc"]}\n')
    results.append(r)
    per_fold_auroc[name] = fold_auc

CNN only
  AUROC 0.8123   AUPRC 0.8712   MCC 0.4713
  sens 0.7083   spec 0.7736   balanced acc 0.7409   units 4,542
  per fold [0.8238, 0.8044, 0.7939, 0.7937, 0.8489]

+ tissue, concat
  AUROC 0.8103   AUPRC 0.8682   MCC 0.4597
  sens 0.7090   spec 0.7607   balanced acc 0.7349   units 4,542
  per fold [0.8174, 0.8054, 0.795, 0.7945, 0.8457]

+ tissue, gated (TGO-Net)
  AUROC 0.7929   AUPRC 0.8537   MCC 0.4320
  sens 0.7159   spec 0.7245   balanced acc 0.7202   units 4,542
  gamma per fold [0.9462, -0.9861, 1.0564, 0.4076, -0.9755]  mean +0.0897  sd 0.9013
  per fold [0.8072, 0.7887, 0.7697, 0.7804, 0.834]



In [8]:
# Cell 7 · ablation, with a paired test across the shared folds
from scipy import stats

tab = pd.DataFrame(results)[['name','auroc','auprc','mcc',
                             'sensitivity','specificity']]
print('ABLATION  (every row an actual run)')
print(tab.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

if len(results) > 1:
    base_all = np.array(per_fold_auroc['CNN only'])
    for name in per_fold_auroc:
        if name == 'CNN only': continue
        other_all = np.array(per_fold_auroc[name])
        # a fold can be NaN if its test slice had only one class present.
        # Exclude those pairs rather than letting NaN propagate silently
        # through the mean, t-test and CI.
        ok = ~(np.isnan(base_all) | np.isnan(other_all))
        n_excluded = int((~ok).sum())
        base, other = base_all[ok], other_all[ok]

        print(f'\n{name} minus CNN only:')
        if n_excluded:
            print(f'  {n_excluded} of {N_FOLDS} folds excluded '
                  f'(single-class test slice, AUROC undefined)')
        if len(base) < 2:
            print(f'  fewer than 2 usable folds ({len(base)}); '
                  f'paired test not meaningful, skipped')
            continue

        d = other - base
        if d.std() > 1e-12:
            t, p = stats.ttest_rel(other, base)
            ci = stats.t.interval(0.95, len(d) - 1, loc=d.mean(),
                                  scale=stats.sem(d))
        else:
            # every fold gave the identical difference: p is undefined by
            # a t-test with zero variance, not "significant" or "zero"
            t, p, ci = float('nan'), float('nan'), (d.mean(), d.mean())
            print(f'  all {len(d)} usable folds gave an identical '
                  f'difference; t-test undefined (zero variance)')

        print(f'  mean AUROC difference {d.mean():+.4f}  (n={len(d)} folds)')
        print(f'  paired t-test p = {p:.3f}' if not np.isnan(p) else
              f'  paired t-test p = undefined')
        print(f'  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]'
              f'{"  spans zero" if ci[0] < 0 < ci[1] else ""}')
        print(f'  folds won by CNN only: {int((base > other).sum())} of {len(d)}')

    tgo = next((r for r in results if 'gated' in r['name']), None)
    if tgo and 'gamma_mean' in tgo:
        print(f'\nlearned gate: mean {tgo["gamma_mean"]:+.4f}, '
              f'sd {tgo["gamma_sd"]:.4f}')
        signs = [np.sign(g) for g in tgo['gamma_per_fold']]
        flips = sum(1 for s in signs if s != signs[0])
        print(f'  sign changes across folds: {flips} of {N_FOLDS}')
        if abs(tgo['gamma_mean']) < tgo['gamma_sd']:
            print('  sd exceeds the mean: the gate never settled.')
            print('  Report as a negative result, not a small positive effect.')

ABLATION  (every row an actual run)
                     name  auroc  auprc    mcc  sensitivity  specificity
                 CNN only 0.8123 0.8712 0.4713       0.7083       0.7736
         + tissue, concat 0.8103 0.8682 0.4597       0.7090       0.7607
+ tissue, gated (TGO-Net) 0.7929 0.8537 0.4320       0.7159       0.7245

+ tissue, concat minus CNN only:
  mean AUROC difference -0.0013  (n=5 folds)
  paired t-test p = 0.428
  95% CI [-0.0055, +0.0029]  spans zero
  folds won by CNN only: 2 of 5

+ tissue, gated (TGO-Net) minus CNN only:
  mean AUROC difference -0.0169  (n=5 folds)
  paired t-test p = 0.001
  95% CI [-0.0222, -0.0116]
  folds won by CNN only: 5 of 5

learned gate: mean +0.0897, sd 0.9013
  sign changes across folds: 2 of 5
  sd exceeds the mean: the gate never settled.
  Report as a negative result, not a small positive effect.


In [9]:
# Cell 8 · the comparison that makes this a control
# Same architectures, same preprocessing, same protocol, same code.
# Only the label source differs.
sev_path = OUT / 'results_severity.json'
best = max(results, key=lambda r: r['auroc'])

print('=' * 62)
print('CONTROL EXPERIMENT: what changes when the labels are real')
print('=' * 62)
if sev_path.exists():
    sev = json.load(open(sev_path))
    sev_best = max(sev['configs'], key=lambda r: r['qwk'])
    print(f'  derived colour labels (nb 06)')
    print(f'    best QWK {sev_best["qwk"]:.4f}   '
          f'severe-vs-rest F1 {sev_best["severe_vs_rest_f1"]:.4f}')
    print(f'  expert infection labels (this notebook)')
    print(f'    best AUROC {best["auroc"]:.4f}   MCC {best["mcc"]:.4f}')
    print()
    print('  The pipeline is unchanged between these two rows. The only')
    print('  difference is where the labels came from. That isolates label')
    print('  provenance as the cause, and rules out model capacity,')
    print('  resolution, preprocessing and training configuration.')
else:
    print(f'  results_severity.json not found; run notebook 06 for the')
    print(f'  side-by-side comparison. This notebook alone gives:')
    print(f'    best AUROC {best["auroc"]:.4f}   MCC {best["mcc"]:.4f}')

CONTROL EXPERIMENT: what changes when the labels are real
  derived colour labels (nb 06)
    best QWK 0.9458   severe-vs-rest F1 0.9930
  expert infection labels (this notebook)
    best AUROC 0.8123   MCC 0.4713

  The pipeline is unchanged between these two rows. The only
  difference is where the labels came from. That isolates label
  provenance as the cause, and rules out model capacity,
  resolution, preprocessing and training configuration.


In [10]:
# Cell 9 · save and summarise
with open(OUT / 'results_infection.json', 'w') as f:
    json.dump(dict(configs=results,
                   n_images=int(len(X)),
                   n_units=int(len(np.unique(unit))),
                   n_patients=int(idx.patient_id.nunique()),
                   class_balance=dict(infected=n_pos, not_infected=n_neg),
                   tissue_available=bool(HAVE_TISSUE)), f, indent=2)
tab.to_csv(OUT / 'ablation_infection.csv', index=False)
print(f'wrote {(OUT / "results_infection.json").resolve()}')
print(f'wrote {(OUT / "ablation_infection.csv").resolve()}')

print('\n' + '=' * 58)
print('STAGE 07 COMPLETE')
print('=' * 58)
print(f'  {len(CONFIGS)} configurations x {N_FOLDS} folds, all executed')
print(f'  scored per photograph ({len(np.unique(unit)):,} units), '
      f'not per image ({len(X):,})')
print(f'  best: {best["name"]} at AUROC {best["auroc"]:.4f}')
if not HAVE_TISSUE:
    print('  tissue ablation FLAGGED: dfuc_tissue.csv absent')
print('\nnext: 08_gradcam.ipynb')

wrote /Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/results_infection.json
wrote /Users/mustaphanasser/Documents/computer_application/DFU_project/notebook2_outputs/ablation_infection.csv

STAGE 07 COMPLETE
  3 configurations x 5 folds, all executed
  scored per photograph (4,542 units), not per image (5,372)
  best: CNN only at AUROC 0.8123

next: 08_gradcam.ipynb
